In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
target_df=spark.table('voltgrid.silver.stations')

In [0]:
df=spark.table('voltgrid.bronze.stations')

In [0]:
source_df=df.withColumn('rnk', F.row_number().over(Window.partitionBy('station_id').orderBy(F.col('ingestion_time').desc()))).filter(F.col('rnk')==1).drop('rnk').withColumn('operator',F.upper(F.col('operator'))).withColumn('city',F.upper(F.col('city'))).withColumn('state_code',F.upper(F.col('state_code'))).withColumn('charger_type',F.upper(F.col('charger_type'))).drop("source_path","ingestion_time","rnk")

## `changed records` Get Changed Records

In [0]:
hash_keys=['name','state_code','city','latitude','longitude','charger_type','max_power_kw','num_connectors','operator','commissioned_date']
source_df=source_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))
target_df=target_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))

In [0]:
history_records=target_df.filter(F.col("active_flg")=="N").drop("hash_key")

In [0]:
target_df=target_df.filter(F.col("active_flg")=="Y")

In [0]:
changed_records=target_df.alias("tr").join(source_df.alias("sr"),F.col("tr.station_id")==F.col("sr.station_id"),'inner').filter(F.col("tr.hash_key") !=F.col("sr.hash_key")).select("tr.*").withColumn('active_flg',F.lit('N')).withColumn('end_date',F.current_date()).drop("hash_key")

In [0]:
target_df=target_df.drop("hash_key")
source_df=source_df.drop("hash_key")

In [0]:
old_records=target_df.alias("tr").join(changed_records.alias("sr"),F.col("tr.station_id")==F.col("sr.station_id"),'left_anti')

In [0]:
new_records=source_df.alias("sr").join(target_df.alias("tr"),F.col("sr.station_id")==F.col("tr.station_id"),'left_anti').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
changed_records_val=source_df.alias("sr").join(changed_records.alias("tr"),F.col("sr.station_id")==F.col("tr.station_id"),'left_semi').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
total_records = (
    history_records.unionByName(changed_records_val)
    .unionByName(new_records)
    .unionByName(old_records)
    .unionByName(changed_records)
)

In [0]:
total_records.display()

In [0]:
total_records.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("voltgrid.silver.stations")